In [1]:
# ===== 기본 세트 (거의 항상 필요) =====
import pandas as pd                    # 데이터프레임 다루기 (표 형태 데이터 처리)
import numpy as np                     # 수치 계산, 배열 연산
import matplotlib.pyplot as plt        # 기본 시각화
import seaborn as sns                  # matplotlib 기반의, 더 예쁘고 간결한 통계 시각화

# ===== 파일/경로/시스템 =====
import os                              # 파일 경로 확인, 파일 존재 여부(os.path.exists) 등
from pathlib import Path               # 폴더/파일 경로를 객체로 다루기 (work_dir / "파일명")
import joblib                          # 학습된 scaler/encoder를 파일로 저장하고 재사용
import platform                        # OS 종류 확인 (Windows/Mac/Linux 구분 필요할 때)
import warnings                        # 경고 메시지 제어(숨기기/표시)

# ===== 벤치마크/성능 측정 =====
import time                            # 코드 실행 시간 측정 (time.perf_counter())
import psutil                          # 메모리 사용량 측정 (rss_mb 등)
import gc                              # 메모리 강제 정리(garbage collection)

# ===== 결측치 탐색 시각화 =====
import missingno as msno               # 결측치 위치(matrix), 관계(heatmap) 시각화

# ===== 인터랙티브 시각화 =====
import plotly.express as px            # 마우스 hover/zoom 가능한 인터랙티브 그래프
import plotly.io as pio                 # plotly 렌더러(출력 방식) 설정

# ===== 대용량 데이터 처리 =====
import polars as pl                     # pandas보다 빠른 대용량 데이터 처리 (Rust 기반)

# ===== 통계 검증 =====
import statsmodels.api as sm            # 회귀분석, 다중공선성(Cond. No.) 등 통계 검증

# ===== 한글 폰트 설정 =====
import matplotlib.font_manager as fm    # 그래프에 한글 폰트 적용 시 필요

In [2]:
elderly_alone = pd.read_csv("경기도 성남시_독거노인_현황_20260605.csv")
population = pd.read_csv("경기도 성남시_인구및세대_현황_20260630.csv")
welfare_center = pd.read_csv("경기도 성남시_노인_종합복지관_현황_20260603.csv")
single_household = pd.read_csv("경기도 성남시_1인세대_현황_20260430.csv")

In [8]:
dataframes = {
    "elderly_alone": elderly_alone,
    "population": population,
    "welfare_center": welfare_center,
    "single_household": single_household
}

for name, df in dataframes.items():
    print("=" * 60)
    print(f"[{name}] info")
    print("=" * 60)
    df.info()
    print()

[elderly_alone] info
<class 'pandas.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   구분       378 non-null    str  
 1   동        378 non-null    str  
 2   연령       378 non-null    str  
 3   남        378 non-null    int64
 4   여        378 non-null    int64
 5   데이터기준일자  378 non-null    str  
dtypes: int64(2), str(4)
memory usage: 31.1 KB

[population] info
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   구별         50 non-null     str  
 1   동          50 non-null     str  
 2   인구수_계      50 non-null     int64
 3   인구수_남      50 non-null     int64
 4   인구수_여      50 non-null     int64
 5   19세 이상_계   50 non-null     int64
 6   19세 이상_남   50 non-null     int64
 7   19세 이상_여   50 non-null     int64
 8   65세 이상_계   50 non-null     int64
 9   65세 이상_남자  50 non

In [44]:
population_copy = population[["구별", "동", "인구수_계", "인구수_남", "인구수_여", "65세 이상_계", "65세 이상_남자", "65세 이상_여자"]].copy()
population_copy.head(10)

,구별,동,인구수_계,인구수_남,인구수_여,65세 이상_계,65세 이상_남자,65세 이상_여자
0,수정구,신흥1동,11914,6370,5544,3212,1413,1799
1,수정구,신흥2동,31566,15329,16237,5467,2509,2958
2,수정구,신흥3동,10616,5711,4905,2682,1271,1411
3,수정구,태평1동,13475,7031,6444,3695,1623,2072
4,수정구,태평2동,13560,6856,6704,3465,1492,1973
5,수정구,태평3동,11457,5884,5573,3128,1405,1723
6,수정구,태평4동,10767,5541,5226,2701,1197,1504
7,수정구,수진1동,10605,5656,4949,3239,1458,1781
8,수정구,수진2동,14273,7211,7062,4123,1873,2250
9,수정구,단대동,14090,6891,7199,3415,1473,1942


In [45]:
single_household_copy = single_household[["구별", "동별", "1인세대수_계", "1인세대수_남", "1인세대수_여", "65세 이상(계)", "65세 이상(남)", "65세 이상(여)"]].copy()
single_household_copy.head(10)

,구별,동별,1인세대수_계,1인세대수_남,1인세대수_여,65세 이상(계),65세 이상(남),65세 이상(여)
0,수정구,신흥1동,4638,2702,1936,1457,625,832
1,수정구,신흥2동,2540,1102,1438,823,224,599
2,수정구,신흥3동,3986,2460,1526,1109,531,578
3,수정구,태평1동,5091,2881,2210,1385,542,843
4,수정구,태평2동,4025,2171,1854,1234,442,792
5,수정구,태평3동,3285,1847,1438,1100,454,646
6,수정구,태평4동,3212,1692,1520,1041,381,660
7,수정구,수진1동,4152,2389,1763,1540,652,888
8,수정구,수진2동,3704,1985,1719,1233,495,738
9,수정구,단대동,2490,1189,1301,894,278,616


In [53]:
population_copy["구별"] = population_copy["구별"].str.strip()
population_copy["동"] = population_copy["동"].str.strip()
single_household_copy["구별"] = single_household_copy["구별"].str.strip()
single_household_copy["동별"] = single_household_copy["동별"].str.strip()

population_single_household = pd.merge(population_copy, single_household_copy, left_on = ["구별", "동"], right_on = ["구별", "동별"], how = "left")
population_single_household.head(100)

,구별,동,인구수_계,인구수_남,인구수_여,65세 이상_계,65세 이상_남자,65세 이상_여자,동별,1인세대수_계,1인세대수_남,1인세대수_여,65세 이상(계),65세 이상(남),65세 이상(여)
0,수정구,신흥1동,11914,6370,5544,3212,1413,1799,신흥1동,4638,2702,1936,1457,625,832
1,수정구,신흥2동,31566,15329,16237,5467,2509,2958,신흥2동,2540,1102,1438,823,224,599
2,수정구,신흥3동,10616,5711,4905,2682,1271,1411,신흥3동,3986,2460,1526,1109,531,578
3,수정구,태평1동,13475,7031,6444,3695,1623,2072,태평1동,5091,2881,2210,1385,542,843
4,수정구,태평2동,13560,6856,6704,3465,1492,1973,태평2동,4025,2171,1854,1234,442,792
5,수정구,태평3동,11457,5884,5573,3128,1405,1723,태평3동,3285,1847,1438,1100,454,646
6,수정구,태평4동,10767,5541,5226,2701,1197,1504,태평4동,3212,1692,1520,1041,381,660
7,수정구,수진1동,10605,5656,4949,3239,1458,1781,수진1동,4152,2389,1763,1540,652,888
8,수정구,수진2동,14273,7211,7062,4123,1873,2250,수진2동,3704,1985,1719,1233,495,738
9,수정구,단대동,14090,6891,7199,3415,1473,1942,단대동,2490,1189,1301,894,278,616


In [ ]:
population_single_household[

In [6]:
elderly_alone.head()

,구분,동,연령,남,여,데이터기준일자
0,수정구,신흥1동,65 - 69,198,263,2026-06-05
1,수정구,신흥1동,70 - 74,178,200,2026-06-05
2,수정구,신흥1동,75 - 79,80,163,2026-06-05
3,수정구,신흥1동,80 - 84,53,106,2026-06-05
4,수정구,신흥1동,85 - 89,12,76,2026-06-05


In [7]:
welfare_center.head()

,복지관명,소재지,설치일자,시설면적(제곱미터),시설규모2,주요사업,전화번호,데이터기준일자
0,수정노인종합복지관,경기도 성남시 수정구 수정남로 268번길 28(산성동),1997-09-25,4493,지하1+지상4,노인여가복지 프로그램+경로식당,031-731-3393,2026-06-03
1,수정중앙노인종합복지관,경기도 성남시 수정구 성남대로 1480번길 38(복정동),2004-04-26,3260,지하1+지상4,노인여가복지 프로그램+경로식당,031-752-3366,2026-06-03
2,중원노인종합복지관,경기도 성남시 중원구 제일로35번길 51(성남동),2007-08-13,10780,지하3+지상4,노인여가복지 프로그램+경로식당,031-751-7450,2026-06-03
3,황송노인종합복지관,경기도 성남시 중원구 금상로 132(상대원동),2010-12-04,2723,지하1+지상3,노인여가복지 프로그램+경로식당,031-731-5520,2026-06-03
4,분당노인종합복지관,경기도 성남시 분당구 불정로 50(정자동),2008-01-11,9319,지하2+지상3,노인여가복지 프로그램+경로식당,031-785-9200,2026-06-03
